# Batch processing a folder with custom parameters

This example runs the pyKrait pipeline on every video in a folder using a single `AnalysisParameters` configuration that you define here.

Each video is processed independently and its results (CSVs, masks, projections, plots) are written to an `Analysis_<video_name>` subfolder next to the source file. After the run, a single overview CSV is created at the root of the folder.

In [ ]:
from pykrait.pipeline.pipeline import BatchExperiment, AnalysisParameters
from pykrait.io.files import concat_analysis_files

## 1. Configure the input folder

Set `folder` to the directory that contains your calcium-imaging videos. All files in this folder matching `extension` will be analyzed.

If you trained or downloaded a custom Cellpose model, point `cellpose_model_path` at the file. Use `"cpsam"` to fall back to the built-in Cellpose-SAM model.

Set `recursive=True` to also search subfolders, and use `filter_string` / `exclude_string` to analyze only some of the acquisitions in them — a pattern with `*`, `?` or `[]` has to match the whole file name, anything else is matched as a substring.

In [ ]:
folder = "/path/to/your/videos"
extension = ".czi"  # also supports ".tif" / ".tiff"
cellpose_model_path = "cpsam"  # or "/path/to/custom_cellpose_model"

recursive = False     # True also searches all subfolders
filter_string = None  # e.g. "*_timeseries.czi" to only analyze the timelapses
exclude_string = None # e.g. "_plane" to skip matching files

In [ ]:
# List the selected videos before starting a long batch
from pykrait.io.files import get_files_from_folder

get_files_from_folder(
    folder,
    extension=extension,
    filter_string=filter_string,
    exclude_string=exclude_string,
    recursive=recursive,
)

## 2. Define the analysis parameters

Every field of `AnalysisParameters` is optional and falls back to a sensible default. Times are given in seconds and distances in micrometers; pyKrait converts to frames/pixels internally using the metadata of each video.

Lateral (xy) motion correction is **disabled by default**. Setting `motion_correction=True` aligns every frame to a template before the projection and the intensity extraction and writes the corrected video to `Analysis_<video_name>/<video_name>_motion_corrected.tif` (an uncompressed ImageJ TIFF that Fiji and napari open directly). A re-run picks that file up instead of correcting again, so delete it if you want to change the correction settings.

In [ ]:
params = AnalysisParameters(
    # Motion correction (optional, disabled by default)
    motion_correction=False,               # if True, every frame is aligned to a template before segmentation and intensity extraction
    motion_correction_method="warp_grid",  # "global_shift" (rigid translation), "warp_grid" (mesh based optical flow) or "warp_polynomial_fit" (polynomial fit of the optical flow)
    motion_correction_channel=1,           # channel the motion is estimated on, works best on a channel with stationary labeling; single-channel videos fall back to channel 0
    motion_correction_template_frames=10,  # number of frames at the beginning of the recording that are averaged into the template
    motion_correction_average_frames=3,    # number of frames averaged before the motion is estimated, the correction itself is applied to the single frame
    motion_correction_template_filter=0.0, # factor to update the template with every corrected frame, 0 keeps the template fixed
    motion_correction_grid_size=10,        # number of mesh cells along each axis, only used for "warp_grid"
    motion_correction_polynomial_degree=2, # degree of the fitted polynomial, only used for "warp_polynomial_fit"
    motion_correction_mask_fov=False,      # if True, pixels that left the field of view in any frame are set to 0 in the whole video
    motion_correction_drop_reference_channel=False, # if True, the reference channel is left out of the saved corrected video, halving its size
    # Channels
    intensity_channel=0,                   # channel the intensity traces are extracted from, given for the original video even if the motion correction drops its reference channel
    # Segmentation
    tproj_type="std",            # "std" or "sum" projection across time
    CLAHE_normalize=True,        # contrast-enhance the projection before Cellpose
    cellpose_model_path=cellpose_model_path,
    frame_interval=None,         # seconds, if None, it is read from the metadata of the video, or defaults to 1.0 seconds if the metadata has none. A given value is used if it differs from the metadata by more than 0.5%, with a warning
    # Detrending
    sinc_filter_window=300,      # cutoff period for the sinc filter in seconds, usually half of the acquisition time
    # Peak detection
    peak_min_width=1.0,          # seconds
    peak_max_width=80.0,         # seconds
    peak_prominence=1500,        # artificial units, see https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.peak_prominences.html
    peak_min_height=800,         # artificial units, minimum value of a peak
    # Periodicity
    periodicity_method="cutoff", # "cutoff" or "quantile", if "cutoff", it will use the std_threshold and cov_threshold to identify periodic cells, if "quantile", it will use the quantiles of the experimental distribution of std and cov to identify periodic cells based on the periodicity_quantiles parameter
    std_threshold=15.0,          # if "cutoff", standard deviation of peak-peak interval in seconds, if "quantile" the quantile (must be between 0 and 1)
    cov_threshold=0.2,           # if "cutoff", coefficient of variation of peak-peak interval, if "quantile" the quantile (must be between 0 and 1)
    # Synchronicity
    time_window=10.0,            # Time in seconds in which the peaks of neighbouring cells are still considered synchronous
    neighbour_tolerance=10.0,    # pixels, maximum distance between two cells to be considered neighbours for the synchronicity analysis
)

## 3. Run the batch

`BatchExperiment(..., load_params=False)` applies the same `params` object to every video. Set `skip_cellpose=True` if cell masks already exist on disk and you want to re-use them; set `skip_intensity_extraction=True` to also re-use cached `*_raw_intensities.csv.zst` files.

A video that fails does not stop the batch: it is listed with its error message in the `error` column of `experiment.results`. Set `raise_errors=True` to stop at the first failure instead, and `seed` to make the random controls of the synchronicity analysis reproducible.

Each video is read twice, once for the motion correction or the projection and once for the intensities. `video_cache` decides where the intensity channel is kept in between so the second read is spared: `"auto"` keeps it in memory if it takes at most half of the available memory and otherwise in a temporary file on disk, `"memory"`, `"disk"` and `"none"` force a location. Set `save_temp_file=False` if no temporary file may be written, e.g. on a nearly full drive; videos that do not fit into memory are then read twice.

In [ ]:
experiment = BatchExperiment(
    folder=folder,
    params=params,
    extension=extension,
    filter_string=filter_string,
    exclude_string=exclude_string,
    recursive=recursive,
    load_params=False,
)

experiment.run(
    skip_cellpose=False,
    skip_intensity_extraction=False,
    raise_errors=False,      # True stops at the first video that fails
    seed=None,               # e.g. 42 for reproducible random controls
    video_cache="auto",      # "auto", "memory", "disk" or "none"
    save_temp_file=True,     # False never writes a temporary file of the video
)

## 4. Generate an overview CSV

`concat_analysis_files` walks the folder tree and merges all per-video result files into a single overview CSV at the root of the folder.

In [ ]:
concat_analysis_files(folder, filetype="output")
concat_analysis_files(folder, filetype="parameters")